In [ ]:
import os
import sys
import json
import copy
import importlib
import numpy as np
import pandas as pd
import torch

In [ ]:
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
with open(os.path.join(BASE_DIR, "config.json")) as f:
    config = json.load(f)
with open(os.path.join(BASE_DIR, "context_config.json")) as f:
    context_config = json.load(f)

SEED = config["seed"]
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR = config["paths"]["results_dir"]
EMOTIONS, IN_DIMS = config["emotions"], context_config["in_dims"]
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)
print(f"Device: {device} | dims: {IN_DIMS} | speaker mode: {context_config['speaker_mode']}")

In [ ]:
MODULE_SRC = r'''
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from meld_context import MultimodalContextEncoder, MELDDialogueDataset, load_split_features


FUSION_MODES = ("gated", "reliability_gated", "reliability_only", "mean",
                "poe_gated", "poe_reliability", "poe_reliability_only", "poe_mean")

U_MEASURES = ("var", "mi", "ent")  
                                    
U_NORMS = ("mean", "zscore", "none")


def make_head(d_in, d_hidden, num_classes, dropout):
    return nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_hidden, num_classes))


class ContextFusionModel(nn.Module):
    """Per-modality context encoders -> per-modality heads (used for uncertainty + auxiliary loss)
    -> weighted fusion of the contextualised modality features -> emotion classifier.

    Uncertainty of modality m (per utterance):
        u_raw_m = (1/C) * sum_c Var_{t=1..T} P_t(y=c | x_m)  (population variance over T MC-dropout passes)
        u_m     = normalise(u_raw_m) per modality using TRAIN-set statistics (see U_NORMS)
    mc_scope="head": dropout is active only in modality m's head (cheap; default)
    mc_scope="full": dropout is active in modality m's context encoder AND head (T full passes)
    u is computed without gradient (detached), in training AND evaluation and for EVERY fusion mode (so baseline and proposed
    consume identical random numbers and the baseline can log comparable uncertainties); lam is a fixed, non-trainable float.
    """

    def __init__(self, in_dims, num_classes=7, modalities=None, fusion="gated", lam=1.0, mc_passes=10,
                 mc_scope="head", u_norm="mean", u_measure="var", u_hinge=False, mask_invalid=False, u_clip=None, d_model=256, n_heads=4, n_layers=2, ff_mult=4,
                 dropout=0.3, num_speaker_ids=13, max_len=64):
        super().__init__()
        if u_measure not in U_MEASURES:
            raise ValueError(f"u_measure must be one of {U_MEASURES}")
        if u_norm not in U_NORMS:
            raise ValueError(f"u_norm must be one of {U_NORMS}")
        if fusion not in FUSION_MODES:
            raise ValueError(f"fusion must be one of {FUSION_MODES}")
        if mc_scope not in ("head", "full"):
            raise ValueError("mc_scope must be 'head' or 'full'")
        self.modalities = list(modalities) if modalities is not None else list(in_dims.keys())
        self.fusion, self.lam, self.mc_passes, self.mc_scope = fusion, float(lam), int(mc_passes), mc_scope
        self.u_norm, self.u_measure, self.u_hinge = u_norm, u_measure, bool(u_hinge)
    
        self.mask_invalid, self.u_clip = bool(mask_invalid), (None if u_clip is None else float(u_clip))
        
        self.num_classes, self.d_model = num_classes, d_model
        self.context = MultimodalContextEncoder(
            in_dims, modalities=self.modalities, d_model=d_model, n_heads=n_heads, n_layers=n_layers,
            ff_mult=ff_mult, dropout=dropout, num_speaker_ids=num_speaker_ids, max_len=max_len)
        M = len(self.modalities)
       
        self.modality_heads = nn.ModuleDict({m: make_head(d_model, d_model // 2, num_classes, dropout) for m in self.modalities})
        self.gate = nn.Linear(M * d_model, M)
        self.fusion_head = make_head(d_model, d_model // 2, num_classes, dropout)
     
        self.register_buffer("u_mu", torch.ones(M))
        self.register_buffer("u_sigma", torch.ones(M))

    @property
    def uses_reliability(self):
        return self.fusion in ("reliability_gated", "reliability_only", "poe_reliability", "poe_reliability_only")

    @torch.no_grad()
    def _mc_probs(self, batch, h):
        """{modality: [T,B,L,C]} class probabilities from T stochastic (dropout-on) passes. No gradients."""
        T, out = self.mc_passes, {}
        for m in self.modalities:
            head, enc = self.modality_heads[m], self.context.encoders[m]
            head_was, enc_was = head.training, enc.training
            head.train(True)
            try:
                if self.mc_scope == "head":
                    hm = h[m].detach()
                    logits = head(hm.unsqueeze(0).expand(T, *hm.shape))
                else:
                    enc.train(True)
                    logits = torch.stack([
                        head(enc(batch["features"][m], batch["speaker_ids"], batch["mask"], batch["valid"][m]))
                        for _ in range(T)])
                out[m] = torch.softmax(logits, dim=-1)
            finally:
                head.train(head_was)
                enc.train(enc_was)
        return out

    def _raw_u(self, batch, h):
        """(u_raw [B,L,M], mc probs {m: [T,B,L,C]})."""
        mc = self._mc_probs(batch, h)
        if self.u_measure == "var":
            u = torch.stack([mc[m].var(dim=0, unbiased=False).mean(dim=-1) for m in self.modalities], dim=-1)
        elif self.u_measure == "ent":                            
            eps = 1e-12
            us = []
            for m in self.modalities:
                pbar = mc[m].mean(0)
                us.append(-(pbar * (pbar + eps).log()).sum(-1) / float(np.log(self.num_classes)))
            u = torch.stack(us, dim=-1)
        else:                                                   
            eps = 1e-12
            us = []
            for m in self.modalities:
                p = mc[m]
                h_mean = -(p.mean(0) * (p.mean(0) + eps).log()).sum(-1)
                mean_h = -(p * (p + eps).log()).sum(-1).mean(0)
                us.append((h_mean - mean_h).clamp_min(0))
            u = torch.stack(us, dim=-1)
        return u, mc

    def normalize_u(self, u_raw):
        if self.u_norm == "mean":
            u = u_raw / self.u_mu
        elif self.u_norm == "zscore":
            u = (u_raw - self.u_mu) / self.u_sigma
        else:
            u = u_raw
        if self.u_clip is not None:
            u = u.clamp(max=self.u_clip)
        return u

    @torch.no_grad()
    def calibrate_uncertainty(self, loader, device, seed=0):
        """Set u_mu / u_sigma to the mean / std of u_raw per modality over the utterances of `loader` (use the TRAIN split),
        computed in eval mode with seeded MC passes. Does not disturb the global RNG state."""
        was_training = self.training
        self.eval()
        s1 = torch.zeros(len(self.modalities), dtype=torch.float64, device=device)
        s2, n = torch.zeros_like(s1), 0
        devices = [torch.cuda.current_device()] if torch.cuda.is_available() else []
        with torch.random.fork_rng(devices=devices):
            torch.manual_seed(seed)
            for batch in loader:
                batch = batch_to_device_(batch, device)
                u_raw, _ = self._raw_u(batch, self.context(batch))
                us = u_raw[batch["mask"]].double()
                s1 += us.sum(0); s2 += (us ** 2).sum(0); n += us.shape[0]
        mu = s1 / n
        sigma = (s2 / n - mu ** 2).clamp_min(0).sqrt()
        self.u_mu.copy_(mu.clamp_min(1e-12).float())
        self.u_sigma.copy_(sigma.clamp_min(1e-12).float())
        self.train(was_training)
        return {"u_mu": self.u_mu.tolist(), "u_sigma": self.u_sigma.tolist(), "n": int(n)}

    def forward(self, batch, return_mc=False, u_override=None):
        """u_override (testing / counterfactuals) replaces the NORMALISED uncertainty that enters the weights."""
        h = self.context(batch)                                               
        mod_logits = {m: self.modality_heads[m](h[m]) for m in self.modalities}
        scores = self.gate(torch.cat([h[m] for m in self.modalities], dim=-1))   

        u_raw, mc = None, None
        if u_override is not None:
            u = u_override
        else:
            u_raw, mc = self._raw_u(batch, h)
            u = self.normalize_u(u_raw)

        if self.u_hinge and u is not None and self.uses_reliability:
            u = torch.relu(u - (1.0 if self.u_norm == "mean" else 0.0))     
        valid_all = torch.stack([batch["valid"][m] for m in self.modalities], dim=-1)          

        if self.fusion.startswith("poe"):
            v = torch.stack([batch["valid"][m] for m in self.modalities], dim=-1).float()        
            logp = torch.stack([torch.log_softmax(mod_logits[m], dim=-1) for m in self.modalities], dim=-2)  
            lg_content = torch.nn.functional.logsigmoid(scores)
            if self.fusion == "poe_gated":
                lg = lg_content
            elif self.fusion == "poe_reliability":
                lg = lg_content - self.lam * u
            elif self.fusion == "poe_reliability_only":
                lg = (-self.lam * u).expand_as(scores)
            else:
                lg = torch.zeros_like(scores)
            g = lg.exp() * v
            gc = lg_content.exp() * v
            logits = (g.unsqueeze(-1) * logp).sum(dim=-2)
            w = g / g.sum(-1, keepdim=True).clamp_min(1e-8)                 
            w_content = gc / gc.sum(-1, keepdim=True).clamp_min(1e-8)
            out = {"logits": logits, "modality_logits": mod_logits, "w": w, "w_content": w_content, "gates": g,
                   "gate_scores": scores, "u": u, "u_raw": u_raw, "h": h}
            if return_mc:
                out["mc_probs"] = mc
            return out

        if self.fusion == "gated":
            wl = scores
        elif self.fusion == "reliability_gated":
            wl = scores - self.lam * u
        elif self.fusion == "reliability_only":
            wl = (-self.lam * u).expand_as(scores)
        else:
            wl = torch.zeros_like(scores)
        sc = scores
        if self.mask_invalid:
            wl = wl.masked_fill(~valid_all, -1e4)
            sc = scores.masked_fill(~valid_all, -1e4)
        w = torch.softmax(wl, dim=-1)
        fused = sum(w[..., i:i + 1] * h[m] for i, m in enumerate(self.modalities))
        out = {"logits": self.fusion_head(fused), "modality_logits": mod_logits, "w": w,
               "w_content": torch.softmax(sc, dim=-1), "gate_scores": scores, "u": u, "u_raw": u_raw, "h": h}
        if return_mc:
            out["mc_probs"] = mc
        return out


def batch_to_device_(batch, device):
    return {k: ({m: t.to(device) for m, t in v.items()} if isinstance(v, dict) else v.to(device)) for k, v in batch.items()}


def compute_loss(out, batch, class_weights=None, aux_weight=0.5):
    """Class-weighted CE on the fused prediction + aux_weight * mean class-weighted CE of the per-modality heads.
    Padded positions (label -100 / mask False) are excluded."""
    mask, y = batch["mask"], batch["labels"][batch["mask"]]
    main = F.cross_entropy(out["logits"][mask], y, weight=class_weights)
    aux = torch.stack([F.cross_entropy(out["modality_logits"][m][mask], y, weight=class_weights)
                       for m in out["modality_logits"]]).mean()
    return main + aux_weight * aux, {"loss_main": float(main.detach()), "loss_aux": float(aux.detach())}


def class_weights_from_labels(labels, num_classes, power=1.0):
    """'Balanced' inverse-frequency weights w_c = (N / (C * n_c))**power (power=1 -> sum_c w_c n_c = N)."""
    counts = np.bincount(np.asarray(labels), minlength=num_classes).astype(np.float64)
    if (counts == 0).any():
        raise ValueError(f"Classes with zero training samples: {np.flatnonzero(counts == 0).tolist()}")
    w = (counts.sum() / (num_classes * counts)) ** power
    return torch.tensor(w, dtype=torch.float32), counts


def load_split_dataset(base_dir, split, in_dims, emotions, speaker_mode="local"):
    """Dataset for one split from the cached features + Phase-6 context manifest (strictly row-aligned)."""
    data_dir, feat_dir = os.path.join(base_dir, "data"), os.path.join(base_dir, "features")
    df = pd.read_csv(os.path.join(data_dir, f"{split}_sent_emo.csv"))
    manifest = pd.read_csv(os.path.join(data_dir, f"context_manifest_{split}.csv"))
    feats, valid = load_split_features(feat_dir, split, in_dims)
    return df, MELDDialogueDataset(df, manifest, feats, valid, emotions, speaker_mode=speaker_mode)
'''

module_path = os.path.join(BASE_DIR, "meld_models.py")
with open(module_path, "w") as f:
    f.write(MODULE_SRC)
import meld_models
importlib.reload(meld_models)
from meld_models import (ContextFusionModel, FUSION_MODES, compute_loss, class_weights_from_labels, load_split_dataset)
from meld_context import make_loader, batch_to_device
print(f"Wrote and imported {module_path}. Fusion modes: {FUSION_MODES}")

In [ ]:
MC_SCOPE = "head"      
MODEL_KWARGS = {
    "in_dims": IN_DIMS, "num_classes": config["num_classes"],
    "d_model": context_config["d_model"], "n_heads": context_config["n_heads"], "n_layers": context_config["n_layers"],
    "ff_mult": context_config["ff_mult"], "dropout": context_config["dropout"],
    "num_speaker_ids": context_config["num_speaker_ids_local"], "max_len": context_config["max_len"],
    "mc_passes": config["mc_dropout_passes"], "mc_scope": MC_SCOPE, "u_norm": "mean",  
}
AUX_WEIGHT = 0.5     

def build_model(fusion, lam=1.0, modalities=None, seed=SEED, **overrides):
    torch.manual_seed(seed)
    return ContextFusionModel(fusion=fusion, lam=lam, modalities=modalities, **{**MODEL_KWARGS, **overrides}).to(device)

baseline = build_model("gated", lam=0.0)
print(f"BASELINE (fusion='gated'): {sum(p.numel() for p in baseline.parameters()):,} parameters")
for name, mod in [("context encoders", baseline.context), ("modality heads", baseline.modality_heads),
                  ("content gate", baseline.gate), ("fusion head", baseline.fusion_head)]:
    print(f"  {name:17s} {sum(p.numel() for p in mod.parameters()):>10,}")

In [ ]:
df_dev, dev_ds = load_split_dataset(BASE_DIR, "dev", IN_DIMS, EMOTIONS, context_config["speaker_mode"])
df_train, train_ds = load_split_dataset(BASE_DIR, "train", IN_DIMS, EMOTIONS, context_config["speaker_mode"])
dev_loader = make_loader(dev_ds, batch_size=16, shuffle=False, seed=SEED)
batch = next(iter(dev_loader))
print(f"train: {len(df_train)} utterances / {len(train_ds)} dialogues | dev: {len(df_dev)} utterances / {len(dev_ds)} dialogues")
print({k: (tuple(v.shape) if torch.is_tensor(v) else '...') for k, v in batch.items() if k != 'dialogue_id'})

In [ ]:
results = []
def check(name, cond, detail=""):
    results.append({"test": name, "passed": bool(cond), "detail": str(detail)})
    print(("PASS  " if cond else "FAIL  ") + name + (f"  [{detail}]" if detail else ""))

def run(model, b, train=False, **kw):
    model.train(train)
    with torch.set_grad_enabled(train):
        return model(batch_to_device(b, device), **kw)

def force_invalid(b, position=1):
    b = copy.deepcopy(b)
    for m in b["valid"]:
        b["valid"][m][0, position] = False
    return b

mask = batch["mask"].to(device)
baseline.eval()
out = run(baseline, batch)
B, L = batch["mask"].shape

check("T1 output shapes", out["logits"].shape == (B, L, 7) and out["w"].shape == (B, L, 3)
      and all(v.shape == (B, L, 7) for v in out["modality_logits"].values()))

w = out["w"][mask]
check("T2 fusion weights are in [0,1] and sum to 1", bool(((w >= 0) & (w <= 1)).all()) and torch.allclose(w.sum(-1), torch.ones_like(w.sum(-1)), atol=1e-6))

torch.manual_seed(1); o1 = run(baseline, batch)
torch.manual_seed(2); o2 = run(baseline, batch)
same_logits = torch.equal(o1["logits"], o2["logits"]) and torch.equal(o1["w"], o2["w"])
mc_differs = (o1["u"] - o2["u"]).abs().max().item() > 0      
check("T3 baseline weights = softmax(content scores); independent of uncertainty/MC randomness",
      same_logits and torch.equal(o1["w"], o1["w_content"]) and mc_differs)

b4 = copy.deepcopy(batch); b4["features"]["audio"] = b4["features"]["audio"] + 3.0
d_w = (run(baseline, b4)["w"] - out["w"]).abs().max().item()
check("T4 content gate responds to modality content", d_w > 1e-4, f"max |dw|={d_w:.4f}")

models_by_mode = {m: build_model(m, lam=1.0) for m in FUSION_MODES}
ref = models_by_mode["gated"].state_dict()
parity = all(list(sd.keys()) == list(ref.keys()) and all(sd[k].shape == ref[k].shape for k in ref)
             for sd in (mm.state_dict() for mm in models_by_mode.values()))
counts = {m: sum(p.numel() for p in mm.parameters()) for m, mm in models_by_mode.items()}
check("T5 identical parameter names/shapes/counts for baseline and proposed", parity and len(set(counts.values())) == 1, counts["gated"])

same_init = all(torch.equal(models_by_mode["gated"].state_dict()[k], models_by_mode["reliability_gated"].state_dict()[k]) for k in ref)
check("T6 same seed -> identical initial weights (paired comparison)", same_init)

cw, counts_cls = class_weights_from_labels(train_ds.labels, config["num_classes"])
fb = force_invalid(batch)
for mode in ("gated", "reliability_gated"):
    mm = models_by_mode[mode]; mm.zero_grad()
    o = run(mm, fb, train=True)
    loss, parts = compute_loss(o, batch_to_device(fb, device), cw.to(device), AUX_WEIGHT)
    loss.backward()
    bad = [n for n, p in mm.named_parameters() if p.grad is None or not torch.isfinite(p.grad).all()]
    check(f"T7 gradients reach all parameters ({mode})", not bad and o["u"] is not None, bad[:3])
    mm.zero_grad()
prop0 = build_model("reliability_gated", lam=0.0)
base_ = build_model("gated", lam=0.0)
torch.manual_seed(7); ob = run(base_, batch, train=True)
torch.manual_seed(7); op = run(prop0, batch, train=True)
check("T7b training-mode forward: proposed(lambda=0) == baseline (identical RNG consumption)",
      ob["u"] is not None and torch.allclose(ob["logits"], op["logits"], atol=1e-6) and torch.allclose(ob["w"], op["w"], atol=1e-7))

baseline.eval()
o = run(baseline, batch)
loss_main, _ = compute_loss(o, batch_to_device(batch, device), cw.to(device), aux_weight=0.0)
lg = torch.log_softmax(o["logits"][mask].double(), -1).cpu().numpy()
y = batch["labels"][batch["mask"]].numpy()
wt = cw.double().numpy()
manual = float((wt[y] * -lg[np.arange(len(y)), y]).sum() / wt[y].sum())
check("T8 class-weighted CE equals manual computation (padding excluded)", abs(manual - float(loss_main)) < 1e-5, f"{manual:.6f} vs {float(loss_main):.6f}")

from sklearn.utils.class_weight import compute_class_weight
sk = compute_class_weight("balanced", classes=np.arange(7), y=train_ds.labels)
check("T9 class weights match sklearn 'balanced' and sum_c w_c*n_c = N",
      np.allclose(cw.numpy(), sk, atol=1e-5) and abs(float((cw.numpy() * counts_cls).sum()) - len(train_ds.labels)) < 1e-3,
      dict(zip(EMOTIONS, np.round(cw.numpy(), 2))))

uni = build_model("gated", modalities=["text"]); bi = build_model("gated", modalities=["text", "audio"])
ou, ob = run(uni, batch), run(bi, batch)
check("T10a unimodal (w=1) and bimodal models run", torch.allclose(ou["w"], torch.ones_like(ou["w"])) and ob["w"].shape[-1] == 2)
om = run(models_by_mode["mean"], batch)
check("T10b 'mean' fusion gives uniform weights 1/M", torch.allclose(om["w"], torch.full_like(om["w"], 1 / 3)))

failed = [r for r in results if not r["passed"]]
with open(os.path.join(RESULTS_DIR, "baseline_verification.json"), "w") as f:
    json.dump({"n_tests": len(results), "n_failed": len(failed), "results": results}, f, indent=2)
assert not failed, f"{len(failed)} verification test(s) failed: {[r['test'] for r in failed]}"
print(f"\nAll {len(results)} verification checks passed.")

In [ ]:
model_config = {"model_kwargs": MODEL_KWARGS, "aux_weight": AUX_WEIGHT, "module": "meld_models.py",
                "fusion_modes": list(FUSION_MODES), "baseline_fusion": "gated", "proposed_fusion": "reliability_gated"}
with open(os.path.join(BASE_DIR, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=4)

baseline_summary = {"model_name": "ContextFusionModel(fusion='gated')", "parameters": sum(p.numel() for p in baseline.parameters()),
                    "mc_scope": MC_SCOPE, "status": "ready_for_training"}
with open(os.path.join(RESULTS_DIR, "baseline_model_summary.json"), "w") as f:
    json.dump(baseline_summary, f, indent=4)
print(baseline_summary)
print("Phase 7 Baseline Fusion completed successfully.")